In [ ]:
# from docling.document_converter import DocumentConverter

# source = "/media/bharath/DATA_8TB1/Bharath/Problem_Statement_2_630a8c8cb6/2_Submission_Input_Data_7e33bbd2e6/2_Submission Input Data/Discharge Summary/Test 1.pdf"

# # 1. Initialize the converter
# converter = DocumentConverter()

# # 2. Convert the entire document in one go (Preserves internal structure)
# result = converter.convert(source)

# # 3. Access individual pages from the result object
# # Docling's 'result.document' contains the structured data
# pages_text = []

# # If you want to iterate through the document's pages as Markdown:
# # Note: Docling usually exports the whole doc, but we can filter by page index
# for page_num, page in result.document.pages.items():
#     # This gets the content specifically associated with this page
#     page_content = result.document.export_to_markdown(page_no=page_num)
#     pages_text.append(page_content)
#     print(f"Processed page {page_num}")

# # Step 3: Preview
# extracted_text = pages_text[0] if pages_text else ""
# print(f"\n✅ Extracted {len(pages_text)} pages successfully!")
# print("First page preview:", extracted_text[:300] + "...")

/home/bharath/.local/lib/python3.10/site-packages/requests/__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (6.0.0.post1)/charset_normalizer (3.4.3) doesn't match a supported version!
  warnings.warn(
[INFO] 2026-02-26 16:10:15,664 [RapidOCR] base.py:22: Using engine_name: torch
[INFO] 2026-02-26 16:10:15,666 [RapidOCR] device_config.py:57: Using GPU device with ID: 0
[INFO] 2026-02-26 16:10:15,675 [RapidOCR] download_file.py:60: File exists and is valid: /home/bharath/.local/lib/python3.10/site-packages/rapidocr/models/ch_PP-OCRv4_det_infer.pth
[INFO] 2026-02-26 16:10:15,676 [RapidOCR] main.py:50: Using /home/bharath/.local/lib/python3.10/site-packages/rapidocr/models/ch_PP-OCRv4_det_infer.pth
[INFO] 2026-02-26 16:10:16,040 [RapidOCR] base.py:22: Using engine_name: torch
[INFO] 2026-02-26 16:10:16,040 [RapidOCR] device_config.py:57: Using GPU device with ID: 0
[INFO] 2026-02-26 16:10:16,042 [RapidOCR] download_file.py:60: File exists and is valid: /home/bharath/.

Processed page 1

✅ Extracted 1 pages successfully!
First page preview: <!-- image -->

## DISCHARGESUMMARY

<!-- image -->

DATE:03/12/2024

Reg.No:X0

<!-- image -->

Mobile

<!-- image -->

Consultant:Dr.

<!-- image -->

OPNo:901

Name

<!-- image -->

## Address

## Chief Complaint

Patientcameforregularhemodialysis

- ·C/OBreathless
- ·C/oMildswellingoverlegs
- ·C...


In [13]:
from docling.document_converter import DocumentConverter
import re
from collections import defaultdict

def extract_metadata(page_text):
    """
    Extract key metadata used for grouping.
    We use Age/Sex + Collection Date (DATE ONLY) as primary fingerprint.
    """

    # Extract Age/Sex
    age_sex_match = re.search(r'Age/Sex\s*:\s*(.*)', page_text)
    age_sex = age_sex_match.group(1).strip() if age_sex_match else "UNKNOWN"

    # Extract ONLY date part (ignore time)
    collection_match = re.search(
        r'Collection Date\s*:\s*([0-9]{2}-[A-Za-z]{3}-[0-9]{4})',
        page_text
    )
    collection_date = collection_match.group(1) if collection_match else "UNKNOWN"

    # Extract Lab No (optional for debugging)
    lab_no_match = re.search(r'Lab No\.\s*:\s*(.*)', page_text)
    lab_no = lab_no_match.group(1).strip() if lab_no_match else "UNKNOWN"

    print(f"Extracted Metadata - Age/Sex: {age_sex}, Collection Date: {collection_date}, Lab No: {lab_no}")

    return age_sex, collection_date

def group_pages_by_patient(pages_text):
    """
    Group pages belonging to same patient using strong fingerprint.
    """

    grouped = defaultdict(list)

    for page_number, page_text in enumerate(pages_text, start=1):
        age_sex, collection_date = extract_metadata(page_text)

        fingerprint = f"{age_sex}_{collection_date}"

        grouped[fingerprint].append((page_number, page_text))

    final_patient_texts = []

    print("\n📌 GROUPING SUMMARY")
    print("=" * 50)

    for patient_index, (key, page_data) in enumerate(grouped.items(), start=1):

        page_numbers = [str(page_num) for page_num, _ in page_data]
        merged_text = "\n\n".join([text for _, text in page_data])

        final_patient_texts.append(merged_text)

        age_sex, collection_date = key.split("_", 1)

        if len(page_numbers) > 1:
            print(f"🟢 Patient {patient_index} ({age_sex}, {collection_date})")
            print(f"   → Merged Pages: {', '.join(page_numbers)}")
        else:
            print(f"🔵 Patient {patient_index} ({age_sex}, {collection_date})")
            print(f"   → Single Page: {page_numbers[0]}")

        print("-" * 50)

    # print(f"\n🎯 Total Unique Patients Identified: {len(final_patient_texts)}\n")

    return final_patient_texts

def process_pdf_and_group_patients(pdf_path):
    """
    MAIN FUNCTION

    Input:
        pdf_path (str)

    Output:
        list of unique patient text blocks
    """

    # Step 1: Convert using Docling
    converter = DocumentConverter()
    result = converter.convert(pdf_path)

    # Step 2: Extract page-wise text
    pages_text = []

    for i, page_num in enumerate(result.document.pages.keys(), start=1):
        page_content = result.document.export_to_markdown(page_no=page_num)
        pages_text.append(page_content)
        print(f"Processed page {i}")

    print(f"\n✅ Extracted {len(pages_text)} pages successfully!")

    # Step 3: Group pages by patient
    unique_patient_texts = group_pages_by_patient(pages_text)

    print(f"\n🎯 Total Unique Patients Identified: {len(unique_patient_texts)}")

    return unique_patient_texts

In [ ]:
import base64

pdf_path = "report_1.pdf"

with open(pdf_path, "rb") as pdf_file:
    pdf_bytes = pdf_file.read()
    pdf_base64 = base64.b64encode(pdf_bytes).decode("utf-8")

unique_patient_lists = process_pdf_and_group_patients(pdf_path)

In [3]:
abdm_extraction_dictionary = {
    "ClinicalArtifacts": {
        "DischargeSummaryRecord": "A Clinical document used to represent the discharge summary record for ABDM HDE data set. It provides a single coherent clinical statement with clinical attestation of a patient's stay.",
        "DiagnosticReportRecord": "A Clinical Artifact representing diagnostic reports, including Radiology and Laboratory reports, that can be shared across the health ecosystem. It provides a single coherent statement of meaning with clinical attestation.",
    },
    "OtherResources": {
        "Patient": "This profile sets minimum expectations for the Patient resource to record, search, and fetch basic demographics and other administrative information about an individual patient.",
        "Practitioner": "This profile sets minimum expectations for the Practitioner resource to record, search, and fetch basic demographics and other administrative information about an individual practitioner.",
        "PractitionerRole": "This profile sets minimum expectations for the PractitionerRole resource to record, search, and fetch the practitioner role for a practitioner within an organization.",
        "Organization": "This profile sets minimum expectations for the Organization resource to record, search, and fetch information about a healthcare organization.",
        "Encounter": "This profile sets minimum expectations for the Encounter resource to record, search, and fetch basic encounter information for an individual patient, such as inpatient or outpatient status.",
        "Condition": "This profile sets minimum expectations for the Condition resource to record, search, and fetch a list of conditions, problems, or diagnoses associated with a patient.",
        "Procedure": "This profile sets minimum expectations for the Procedure resource to record, search, and fetch details of clinical actions or procedures performed on or with a patient.",
        "Observation": "Represents an individual laboratory test and result value, or a finding. It sets minimum expectations for the Observation resource to record, search, and fetch clinical observations associated with a patient.",
        "DiagnosticReportLab": "This profile represents the set of information related to the Laboratory diagnosis report generated by laboratory services like CBC, Lipid Panel, Urinalysis, etc.",
        "DiagnosticReportImaging": "This profile represents the set of information related to the Imaging diagnosis report generated by imaging services like Radiology, Cardiology, or Endoscopy.",
        "ObservationVitalSigns": "This profile sets minimum expectations for the Observation resource to record, search, and fetch vital signs like Blood Pressure, Heart Rate, and Temperature.",
        "ObservationBodyMeasurement": "This profile sets minimum expectations for the Observation resource to record, search, and fetch physical metrics such as Body Weight, Height, and BMI.",
        "ObservationGeneralAssessment": "This profile sets minimum expectations for the ObservationGeneralAssessment to record, search, and fetch the details of the general health assessment or qualitative scores of a patient.",
        "ObservationLifestyle": "This profile sets minimum expectations for the ObservationLifestyle to record, search, and fetch details of the lifestyle of the patient (e.g., smoking or alcohol status).",
        "ObservationPhysicalActivity": "This profile sets minimum expectations for the ObservationPhysicalActivity to record, search, and fetch details of the physical movement and exercise levels of the patient.",
        "ObservationWomenHealth": "This profile sets minimum expectations for the Observation resource to record specific metrics related to obstetric and gynecological history, such as LMP and pregnancy status.",
        "MedicationRequest": "This resource is used to record a patient's medication prescription or order. It sets minimum expectations to record, search, and fetch medications associated with a patient.",
        "MedicationStatement": "Used to record a patient's medication information, specifically medications consumed by the patient in the past, present, or future.",
        "Medication": "This profile sets the minimum expectations for the medication resource in order to store various details about a given medicine (ingredients, form, etc.).",
        "AllergyIntolerance": "Records the risk of harmful or undesirable physiological response unique to an individual associated with exposure to a substance (food, drug, or material).",
        "FamilyMemberHistory": "This profile sets minimum expectations to record, search, and fetch significant health conditions of the patient's relatives for risk assessment.",
        "Immunization": "This profile sets minimum expectations for the Immunization resource to record, fetch, and search immunization history and vaccine administration associated with a patient.",
        "CarePlan": "This profile sets minimum expectations for the CarePlan resource to record, search, and fetch assessment and plan of treatment data associated with a patient.",
        "ServiceRequest": "A record of a request for service such as diagnostic investigations, treatments, or referrals to be performed.",
        "Specimen": "This profile sets minimum expectations for the Specimen resource to record details about a biological sample (blood, urine, etc.) used in diagnostic testing.",
        "ImagingStudy": "Representation of the content produced in a DICOM imaging study, comprising a set of series and instances (images) acquired in a common context.",
        "DocumentReference": "This profile sets minimum expectations for searching and fetching patient documents, including clinical notes, using a reference to a document.",
        "Binary": "This profile sets minimum expectations for the Binary resource to search and fetch the data of a single raw artifact (e.g., PDF or scanned image) in its native format.",
        "Media": "This profile sets minimum expectations for the Media resource to search and fetch media like a photo, video, or audio recording acquired or used in healthcare."
    }
}

In [4]:
from langchain_ollama import ChatOllama

llm = ChatOllama(
    model="qwen2.5:32b", 
    temperature=0,
    # num_predict is the "max tokens" for the output. 
    # Setting this to 8192 prevents the "EOF" error.
    num_predict=8192, 
    # num_ctx is the input memory. 
    # 32k is usually plenty for clinical text.
    num_ctx=32768
)

In [5]:
import json
import re

# Deterministic mapping of mandatory resources based on the selected artifact
def get_must_resources(artifact):
    if artifact == "DiagnosticReportRecord":
        return [
            "DocumentBundle", "DiagnosticReportRecord", "Patient", "Practitioner", 
            "Organization", "DiagnosticReportLab", "Observation", "DocumentReference"
        ]
    elif artifact == "DischargeSummaryRecord":
        return [
            "DocumentBundle", "DischargeSummaryRecord", "Patient", "Encounter", "Practitioner", 
            "Organization", "Condition", "Procedure", "Specimen", "Appointment", 
            "Observation", "DocumentReference"
        ]
    return []

In [6]:
# Dependency graph - what each resource needs
RESOURCE_DEPENDENCIES = {
    "Patient": [],
    "Organization": [],
    "Practitioner": ["Patient"],
    "PractitionerRole": ["Patient", "Practitioner", "Organization"],
    "Encounter": ["Patient", "Practitioner", "PractitionerRole", "Organization"],
    "Observation": ["Patient", "Encounter"],
    "ObservationVitalSigns": ["Patient", "Encounter"],
    "ObservationBodyMeasurement": ["Patient", "Encounter"],
    "Condition": ["Patient", "Encounter"],
    "Procedure": ["Patient", "Encounter", "Practitioner"],
    "DiagnosticReportLab": ["Patient", "Practitioner", "Observation"],
    "DiagnosticReportImaging": ["Patient", "Practitioner", "Observation"],
    "MedicationRequest": ["Patient", "Practitioner"],
    "MedicationStatement": ["Patient"],
    "Medication": ["Patient"],
    "AllergyIntolerance": ["Patient"],
    "DocumentReference": ["Patient"],
    "DischargeSummaryRecord": ["Patient", "Encounter"],  # Composition
    "DiagnosticReportRecord": ["Patient", "DiagnosticReportLab"]  # Composition
}

In [7]:
import json
import uuid
import operator
from typing import TypedDict, List, Dict, Annotated, Any
from langgraph.graph import StateGraph, END
# from langchain_ollama import ChatOllama
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage
import os
from datetime import datetime, timezone

# ---------------- STATE ----------------
class AgentState(TypedDict):
    text: str
    id_registry: Dict[str, Any]
    final_resources: Annotated[List[dict], operator.add]
    rulebook_paths: Dict[str, str]

# ---------------- LLM ----------------
# llm = ChatOllama(model="qwen2.5:latest", temperature=0)
# llm = ChatOllama(model="deepseek-coder-v2", temperature=0)

# ---------------- JSON EXTRACTION ----------------
def extract_json(text: str):
    if not text or not text.strip():
        return None
    
    # Remove markdown code blocks
    text = text.strip()
    if text.startswith("```json"):
        text = text[7:]
    if text.startswith("```"):
        text = text[3:]
    if text.endswith("```"):
        text = text[:-3]
    text = text.strip()
    
    decoder = json.JSONDecoder()
    idx = 0
    
    while idx < len(text):
        try:
            obj, end = decoder.raw_decode(text[idx:])
            if isinstance(obj, str):
                try:
                    obj = json.loads(obj)
                except:
                    pass
            return obj
        except json.JSONDecodeError:
            idx += 1
    return None

# ---------------- NORMALIZE FUNCTIONS ----------------
def ensure_id(resource):
    if not isinstance(resource, dict):
        return resource
    if "id" not in resource or not resource["id"]:
        resource["id"] = str(uuid.uuid4())
    return resource

def normalize_resource_output(res, resource_type):
    """Convert any input to single dict or list of dicts."""
    if isinstance(res, str):
        parsed = extract_json(res)
        if parsed:
            res = parsed
    
    if isinstance(res, dict):
        return [res]
    elif isinstance(res, list):
        return res
    else:
        # Create minimal resource
        return [{
            "resourceType": resource_type,
            "id": str(uuid.uuid4()),
            "meta": {"profile": [f"https://nrces.in/ndhm/fhir/r4/StructureDefinition/{resource_type}"]}
        }]

def get_single_resource(resources_list, resource_type):
    """Get first valid resource from list."""
    for res in resources_list:
        if isinstance(res, dict) and res.get("resourceType") == resource_type:
            return res
    # Return first item or create new
    if resources_list:
        res = resources_list
        if isinstance(res, dict):
            res["resourceType"] = resource_type
            return res
    return {
        "resourceType": resource_type,
        "id": str(uuid.uuid4()),
        "meta": {"profile": [f"https://nrces.in/ndhm/fhir/r4/StructureDefinition/{resource_type}"]}
    }


In [8]:
# ---------------- CORE AGENT FUNCTION ----------------
def run_extraction_agent(state: AgentState, resource_type: str):
    rulebook_path = state['rulebook_paths'].get(resource_type)
     # Load rulebook content
    rulebook_content = ""
    if rulebook_path and os.path.exists(rulebook_path):
        with open(rulebook_path, 'r', encoding='utf-8') as f:
            rulebook_content = f.read()
    
    prompt = f'''
    Extract ONLY a valid HL7 FHIR R4 {resource_type} resource (or array of resources) from the clinical text.

RULEBOOK (ADDITIONAL STRUCTURE GUIDANCE):
{rulebook_content}
    

CLINICAL TEXT:
{state["text"]}

STRICT REQUIREMENTS (NON-NEGOTIABLE):

• Output MUST be valid JSON only  
• Output MUST start with "{{" or "["  
• DO NOT output markdown, comments, explanations, or code fences  
• DO NOT hallucinate or infer missing data  
• Extract ONLY information explicitly present in the clinical text  
• Omit any field whose value is not clearly present  

FHIR + ABDM CONSTRAINTS:

• Conform to HL7 FHIR R4 structure  
• Conform to ABDM / NDHM profiling expectations  
• Include "resourceType" correctly  
• Every resource MUST contain an "id"  
• "id" MUST be a UUID string (RFC-4122 format)  
• Use only fields relevant to {resource_type}  
• DO NOT include empty objects or empty arrays  
• DO NOT include null values  

TERMINOLOGY RULES:

• Clinical concepts → SNOMED CT codes when applicable  
• Laboratory / measurements → LOINC codes when applicable  
• Units → UCUM codes  
• Include proper system URLs:

  SNOMED CT → http://snomed.info/sct  
  LOINC → http://loinc.org  
  UCUM → http://unitsofmeasure.org  

• If no explicit coded value exists in text → use only "text" representation  
• NEVER fabricate codes

REFERENCE & LINKING RULES:

• Use URN UUID references when linking resources:

  "reference": "urn:uuid:<resource-id>"

• Only create references that are explicitly justified by the text  
• DO NOT create imaginary relationships  

DATA ACCURACY RULES:

• Preserve original clinical meaning  
• Preserve numeric precision exactly  
• Preserve dates exactly as written  
• DO NOT normalize, reinterpret, or guess  

OUTPUT FORMAT:

Return ONLY the JSON resource(s) for {resource_type}.
'''

    try:
        response = llm.invoke([HumanMessage(content=prompt)])
        raw_output = response.content.strip()
        print(f"\n🔍 Raw output for {resource_type}:\n{raw_output[:500]}...")
        
        parsed = extract_json(raw_output)
        if parsed:
            return parsed
        
        print(f"⚠️ Could not parse JSON for {resource_type}")
        
    except Exception as e:
        print(f"❌ Error for {resource_type}: {e}")
    
    # Fallback minimal resource
    return [{
        "resourceType": resource_type,
        "id": str(uuid.uuid4()),
        "meta": {"profile": [f"https://nrces.in/ndhm/fhir/r4/StructureDefinition/{resource_type}"]}
    }]


In [9]:

_node_cache = {}

# def create_resource_node(resource_type: str):
#     """Factory function with caching - creates UNIQUE node per resource"""
#     if resource_type in _node_cache:
#         return _node_cache[resource_type]
    
#     def node(state: AgentState):
#         resources = run_extraction_agent(state, resource_type)
#         resources = normalize_resource_output(resources, resource_type)
        
#         if isinstance(resources, list):
#             safe_resources = []
#             for res in resources[:10]:
#                 if isinstance(res, dict):
#                     if res.get("resourceType") != resource_type:
#                         res["resourceType"] = resource_type
#                     res = ensure_id(res)
                    
#                     # Add references to available dependencies
#                     patient_id = state['id_registry'].get('patient_id')
#                     if patient_id and 'subject' in res:
#                         res['subject'] = {'reference': f'urn:uuid:{patient_id}'}
                    
#                     safe_resources.append(res)
#             result = safe_resources
#         else:
#             result = get_single_resource([resources], resource_type)
#             result = ensure_id(result)
            
#             patient_id = state['id_registry'].get('patient_id')
#             if patient_id and 'subject' in result:
#                 result['subject'] = {'reference': f'urn:uuid:{patient_id}'}
        
#         # Register ID(s)
#         if isinstance(result, list):
#             state['id_registry'][f'{resource_type.lower()}_refs'] = [
#                 {'reference': f'urn:uuid:{r["id"]}'} for r in result
#             ]
#         else:
#             state['id_registry'][f'{resource_type.lower()}_id'] = result['id']
        
#         count = len(result) if isinstance(result, list) else 1
#         print(f"✅ {resource_type}: {count}")
#         return {"final_resources": [result] if isinstance(result, list) else [result]}
    
#     node.__name__ = f"{resource_type.lower()}_node"
#     _node_cache[resource_type] = node  # ✅ CACHE THE FUNCTION
#     return node

def create_resource_node(resource_type: str):
    """Factory function with caching - handles MULTIPLE observations + Composition"""
    if resource_type in _node_cache:
        return _node_cache[resource_type]
    
    def node(state: AgentState):
        # ✅ SPECIAL CASE: Clinical Artifacts use Composition structure
        if resource_type in ["DiagnosticReportRecord", "DischargeSummaryRecord"]:
            actual_resource_type = "Composition"
            is_composition = True
        else:
            actual_resource_type = resource_type
            is_composition = False
        
        resources = run_extraction_agent(state, actual_resource_type)
        resources = normalize_resource_output(resources, actual_resource_type)
        
        # ✅ KEEP YOUR ORIGINAL MULTIPLE HANDLING LOGIC
        if isinstance(resources, list):
            safe_resources = []
            max_items = 1 if is_composition else 10  # ✅ Composition=1, Others=10
            for res in resources[:max_items]:
                if isinstance(res, dict):
                    if res.get("resourceType") != actual_resource_type:
                        res["resourceType"] = actual_resource_type
                    
                    # ✅ Composition profile forcing
                    if is_composition:
                        res.setdefault('meta', {})['profile'] = [
                            f"https://nrces.in/ndhm/fhir/r4/StructureDefinition/{resource_type}"
                        ]
                    
                    res = ensure_id(res)
                    
                    # Add patient reference (YOUR ORIGINAL LOGIC)
                    patient_id = state['id_registry'].get('patient_id')
                    if patient_id and 'subject' in res:
                        res['subject'] = {'reference': f'urn:uuid:{patient_id}'}
                    
                    safe_resources.append(res)
            result = safe_resources
        else:
            result = get_single_resource([resources], actual_resource_type)
            result = ensure_id(result)
            
            # Composition profile
            if is_composition:
                result.setdefault('meta', {})['profile'] = [
                    f"https://nrces.in/ndhm/fhir/r4/StructureDefinition/{resource_type}"
                ]
            
            # Patient reference (YOUR ORIGINAL LOGIC)
            patient_id = state['id_registry'].get('patient_id')
            if patient_id and 'subject' in result:
                result['subject'] = {'reference': f'urn:uuid:{patient_id}'}
        
        # ✅ YOUR ORIGINAL ID REGISTRATION LOGIC (handles lists AND singles)
        if isinstance(result, list):
            state['id_registry'][f'{resource_type.lower()}_refs'] = [
                {'reference': f'urn:uuid:{r["id"]}'} for r in result
            ]
        else:
            state['id_registry'][f'{resource_type.lower()}_id'] = result['id']
        
        count = len(result) if isinstance(result, list) else 1
        resource_display = "Composition" if is_composition else resource_type
        print(f"✅ {resource_type}: {count} {resource_display}")
        return {"final_resources": [result] if isinstance(result, list) else [result]}
    
    node.__name__ = f"{resource_type.lower()}_node"
    _node_cache[resource_type] = node
    return node


# ✅ CLEAR CACHE BETWEEN WORKFLOWS (if needed)
def clear_node_cache():
    global _node_cache
    _node_cache = {}


In [10]:
# def assembly_node(state):
#     """Create ABDM-compliant Bundle with Composition entry"""
#     bundle = {
#         "resourceType": "Bundle",
#         "id": str(uuid.uuid4()),
#         "meta": {
#             "profile": ["https://nrces.in/ndhm/fhir/r4/StructureDefinition/DocumentBundle"]
#         },
#         "type": "document",
#         "identifier": {
#             "system": "https://www.abdm.gov.in/bundle",
#             "value": str(uuid.uuid4())
#         },
#         "timestamp": datetime.now(timezone.utc).isoformat(),
#         "entry": []
#     }
    
#     # Add Composition as first entry (clinical artifact)
#     clinical_artifact = state.get('clinical_artifact')
#     for resources in state["final_resources"]:
#         if isinstance(resources, list) and any(r.get('resourceType') == clinical_artifact for r in resources):
#             for comp in resources:
#                 if comp.get('resourceType') == clinical_artifact:
#                     bundle["entry"].insert(0, {
#                         "fullUrl": f"urn:uuid:{comp['id']}",
#                         "resource": comp
#                     })
    
#     # Add all other resources
#     seen_ids = {entry['resource']['id'] for entry in bundle["entry"]}
#     all_resources = state["final_resources"]
    
#     for resources in all_resources:
#         if isinstance(resources, list):
#             for r in resources:
#                 if isinstance(r, dict) and r.get('id') not in seen_ids:
#                     seen_ids.add(r['id'])
#                     bundle["entry"].append({
#                         "fullUrl": f"urn:uuid:{r['id']}",
#                         "resource": r
#                     })
#         elif isinstance(resources, dict) and resources.get('id') not in seen_ids:
#             seen_ids.add(resources['id'])
#             bundle["entry"].append({
#                 "fullUrl": f"urn:uuid:{resources['id']}",
#                 "resource": resources
#             })
    
#     print(f"✅ Bundle created with {len(bundle['entry'])} entries")
#     return {"final_resources": [bundle]}

def assembly_node(state):
    """Your logic + Composition FIRST + DocumentReference LAST"""
    bundle = {
        "resourceType": "Bundle",
        "id": str(uuid.uuid4()),
        "meta": {"profile": ["https://nrces.in/ndhm/fhir/r4/StructureDefinition/DocumentBundle"]},
        "type": "document",
        "identifier": {"system": "https://www.abdm.gov.in/bundle", "value": str(uuid.uuid4())},
        "timestamp": datetime.now(timezone.utc).isoformat(),
        "entry": []
    }
    
    clinical_artifact = state.get('clinical_artifact')
    
    # ✅ STEP 1: YOUR ORIGINAL Composition logic (IMPROVED matching)
    composition_found = False
    for resources in state["final_resources"]:
        if isinstance(resources, list):
            for comp in resources:
                if (comp.get('resourceType') == 'Composition' and 
                    clinical_artifact in [p.split('/')[-1] for p in comp.get('meta', {}).get('profile', [])]):
                    bundle["entry"].insert(0, {
                        "fullUrl": f"urn:uuid:{comp['id']}",
                        "resource": comp
                    })
                    composition_found = True
                    print(f"✅ Composition ({clinical_artifact}) added FIRST")
                    break
            if composition_found:
                break
    
    # ✅ STEP 2: YOUR EXACT OTHER RESOURCES LOGIC
    seen_ids = {entry['resource']['id'] for entry in bundle["entry"]}
    doc_refs = []  # Collect DocumentReference separately
    
    # First pass: collect DocumentReference
    for resources in state["final_resources"]:
        if isinstance(resources, list):
            for r in resources:
                if (isinstance(r, dict) and r.get('resourceType') == 'DocumentReference' and 
                    r.get('id') not in seen_ids):
                    doc_refs.append(r)
        elif (isinstance(resources, dict) and resources.get('resourceType') == 'DocumentReference' and 
              resources.get('id') not in seen_ids):
            doc_refs.append(resources)
    
    # Second pass: add NON-DocumentReference (YOUR EXACT LOGIC)
    for resources in state["final_resources"]:
        if isinstance(resources, list):
            for r in resources:
                if (isinstance(r, dict) and r.get('id') not in seen_ids and 
                    r.get('resourceType') != 'DocumentReference'):
                    seen_ids.add(r['id'])
                    bundle["entry"].append({
                        "fullUrl": f"urn:uuid:{r['id']}",
                        "resource": r
                    })
        elif (isinstance(resources, dict) and resources.get('id') not in seen_ids and 
              resources.get('resourceType') != 'DocumentReference'):
            seen_ids.add(resources['id'])
            bundle["entry"].append({
                "fullUrl": f"urn:uuid:{resources['id']}",
                "resource": resources
            })
    
    # ✅ STEP 3: DocumentReference LAST (NEW)
    for doc_ref in doc_refs:
        seen_ids.add(doc_ref['id'])
        bundle["entry"].append({
            "fullUrl": f"urn:uuid:{doc_ref['id']}",
            "resource": doc_ref
        })
    
    print(f"✅ Bundle: {len([e for e in bundle['entry'] if e['resource']['resourceType']=='Composition'])} Composition + "
          f"{len(doc_refs)} DocumentReference LAST + {len(bundle['entry'])} total")
    return {"final_resources": [bundle]}

In [11]:
def build_dynamic_workflow(clinical_artifact: str, selected_other_resources: List[str], rulebook_paths: Dict[str, str]):
    must_resources = get_must_resources(clinical_artifact)
    selected_other_resources = [res for res in selected_other_resources if res not in must_resources]
    all_resources = list(set(must_resources + selected_other_resources))
    all_resources.append(clinical_artifact)
    
    print(f"📋 Workflow for {clinical_artifact}: {all_resources}")
    
    workflow = StateGraph(AgentState)
    
    # ✅ CREATE NODES - Each resource gets EXACTLY ONE node
    created_nodes = set()
    for resource in all_resources:
        node_name = resource.lower()
        if node_name not in created_nodes:
            node_func = create_resource_node(resource)
            workflow.add_node(node_name, node_func)
            created_nodes.add(node_name)
            print(f"✅ Added node: {node_name}")
    
    # Topological sort (unchanged)
    def topological_sort(resources):
        visited = set()
        order = []
        def visit(resource):
            if resource in visited: return
            visited.add(resource)
            for dep in RESOURCE_DEPENDENCIES.get(resource, []):
                if dep in resources:
                    visit(dep)
            order.append(resource)
        for resource in resources:
            visit(resource)
        return order
    
    resource_order = topological_sort(all_resources)
    print(f"📊 Execution order: {[r.lower() for r in resource_order]}")
    
    # Safe edges
    for i in range(len(resource_order) - 1):
        current = resource_order[i].lower()
        next_node = resource_order[i + 1].lower()
        if current in workflow.nodes and next_node in workflow.nodes:
            workflow.add_edge(current, next_node)
            print(f"➡️  Edge: {current} → {next_node}")
    
    # Assembly
    workflow.add_node("assembly", assembly_node)
    last_node = resource_order[-1].lower()
    if last_node in workflow.nodes:
        workflow.add_edge(last_node, "assembly")
    workflow.add_edge("assembly", END)
    
    if "patient" in workflow.nodes:
        workflow.set_entry_point("patient")
    
    return workflow.compile(), all_resources

In [12]:
def clean_and_reorder_bundle(input_file, output_file):
    # 1. Load the JSON
    with open(input_file, 'r') as f:
        bundle = json.load(f)

    entries = bundle.get("entry", [])
    
    # Identify indices for removal and relocation
    composition_entry = None
    cleaned_entries = []

    for entry in entries:
        resource = entry.get("resource", {})
        res_type = resource.get("resourceType")

        # Task A: Find the Composition to move it later
        if res_type == "Composition":
            composition_entry = entry
        
        # Task B: Identify and skip the fake 'DocumentBundle' resource
        elif res_type == "DocumentBundle":
            print(f"🗑️ Removing invalid 'DocumentBundle' resource (ID: {resource.get('id')})")
            continue
            
        else:
            cleaned_entries.append(entry)

    # Task C: Reassemble with Composition at the very beginning
    if composition_entry:
        final_entries = [composition_entry] + cleaned_entries
        bundle["entry"] = final_entries
        # print("✅ Success: Composition moved to index 0.")
    else:
        bundle["entry"] = cleaned_entries
        # print("⚠️ Warning: No Composition resource was found to relocate.")

    # 3. Save the corrected JSON
    with open(output_file, 'w') as f:
        json.dump(bundle, f, indent=2)
    
    # print(f"💾 Corrected file saved as: {output_file}")


In [30]:
import json

def document_reference_node(input_file, output_file, pdf_base64):

    with open(input_file, 'r') as f:
        bundle = json.load(f)

    updated = False

    for entry in bundle.get("entry", []):
        resource = entry.get("resource", {})

        if resource.get("resourceType") == "DocumentReference":

            # If content does NOT exist → create it
            if "content" not in resource or not resource["content"]:

                resource["content"] = [
                    {
                        "attachment": {
                            "contentType": "application/pdf",
                            "data": pdf_base64
                        }
                    }
                ]
                print(resource["content"][0]['attachment']['data'])
                # print("Created new content block")

            else:
                # Content exists → update attachment
                attachment = resource["content"][0].setdefault("attachment", {})

                attachment["contentType"] = "application/pdf"
                attachment["data"] = pdf_base64
                print(attachment['data'])
                # print("Updated existing content block")

    with open(output_file, 'w') as f:
        json.dump(bundle, f, indent=2)

In [28]:
# Updated initial state
def run_abdm_pipeline(extracted_text: str, clinical_artifact: str, selected_other_resources: List[str], patient_index: int):
    # Complete rulebook paths (add all your paths)
    rulebook_paths = {
        # Your existing paths + add for all resources
        "Patient": "/media/miglab/DATA_20TB1/NHCX/rulebooks_updated/StructureDefinition-Patient_updated.json",
        "Organization": "/media/miglab/DATA_20TB1/NHCX/rulebooks_updated/StructureDefinition-Organization_updated.json",
        # ... add all 38 resources
        **{
            res: f"/media/miglab/DATA_20TB1/NHCX/rulebooks_updated/StructureDefinition-{res}_updated.json"
            for res in abdm_extraction_dictionary["OtherResources"]
        }
    }
    
    initial_state = {
        "text": extracted_text,
        "clinical_artifact": clinical_artifact,
        "id_registry": {},
        "final_resources": [],
        "rulebook_paths": rulebook_paths
    }
    
    # Build and run dynamic workflow
    app, used_resources = build_dynamic_workflow(clinical_artifact, selected_other_resources, rulebook_paths)
    
    print(f"🚀 Starting FHIR Bundle Generation for Patient {patient_index}...")
    final_output = app.invoke(initial_state)
    bundle = final_output['final_resources'][-1]
    # Save output
    filename = f"FHIR_BUNDLE_{clinical_artifact}_Patient{patient_index}.json"
    with open(filename, "w") as f:
        json.dump(bundle, f, indent=2)

    clean_and_reorder_bundle(f"FHIR_BUNDLE_{clinical_artifact}_Patient{patient_index}.json", f"FHIR_BUNDLE_{clinical_artifact}_Patient{patient_index}.json")
    document_reference_node(f"FHIR_BUNDLE_{clinical_artifact}_Patient{patient_index}.json", f"FHIR_BUNDLE_{clinical_artifact}_Patient{patient_index}.json")
    
    print(f"\n SUCCESS! FHIR Bundle saved as {filename}")
    print(f" Resources processed: {used_resources}")
    print(f" Bundle entries: {len(bundle['entry'])}")
    
    return bundle

# # Usage example (from your LLM classification)
# clinical_artifact = "DiagnosticReportRecord"  # From your LLM
# must_resources = get_must_resources(clinical_artifact)
# selected_other_resources = ["ObservationVitalSigns"]  # From your LLM

for idx, extracted_text in enumerate(unique_patient_lists, start=1):
    print(f"\n--- Patient {idx} ---")
    extracted_text = unique_patient_lists[idx-1]

    prompt = f"""
    ACT AS an expert ABDM FHIR Data Architect.

    TASK:
    1. Analyze the [Extracted Text] and select the most appropriate key from [ClinicalArtifacts].
    2. Select any relevant keys from [RemainingResources] that are explicitly mentioned in the text. 
    - DO NOT select resources that are already part of the Mandatory Base for your chosen artifact.
    - The number of selected resources can be zero or more depending on the text content.

    INPUT:
    [Extracted Text]: 
    {extracted_text}

    [Dictionary]:
    {json.dumps(abdm_extraction_dictionary, indent=2)}

    OUTPUT FORMAT:
    Return ONLY a valid JSON object. No pre-amble or markdown blocks.
    {{
        "clinical_artifact": "SelectedKeyFromClinicalArtifacts",
        "selected_other_resources": ["Key1", "Key2", ...]
    }}
    """

    # Invoke the LLM
    response = llm.invoke(prompt)
    raw_output = response.content.strip()

    # Parsing Logic
    try:
        # Clean up potential markdown formatting
        clean_json = re.sub(r'^```json\s*|```$', '', raw_output, flags=re.MULTILINE).strip()
        data = json.loads(clean_json)
        
        # Final Variables
        clinical_artifact = data.get("clinical_artifact", "")
        must_resources = get_must_resources(clinical_artifact)
        selected_other_resources = data.get("selected_other_resources", [])
        selected_other_resources = [res for res in selected_other_resources 
                            if res not in must_resources]
        
        # Logging the results
        print(f"--- Extraction Complete ---")
        print(f"Artifact: {clinical_artifact}")
        print(f"Must Resources (Fixed): {must_resources}")
        print(f"Other Selected Resources: {selected_other_resources}")

    except Exception as e:
        print(f"Error parsing LLM output: {e}")
        print(f"Raw response was: {raw_output}")

    bundle = run_abdm_pipeline(extracted_text, clinical_artifact, selected_other_resources, idx)

print("\n============================")
print("BATCH SUMMARY")
print("============================")
print(f"Total Patients: {len(unique_patient_lists)}")

NameError: name 'List' is not defined

In [32]:
document_reference_node("/media/miglab/DATA_20TB1/NHCX/Diagnostic Report Test 2 Bundles/FHIR_BUNDLE_DiagnosticReportRecord_Patient1.json", "/media/miglab/DATA_20TB1/NHCX/Diagnostic Report Test 2 Bundles/FHIR_BUNDLE_DiagnosticReportRecord_Patient1.json", pdf_base64)

...


In [14]:
import json
with open('fixed_abdm_bundle_discharge_summary.json', 'r') as f:
    bundle_json = json.load(f)
original_text = extracted_text

In [ ]:
import json
import re

# --- REQUIRED PLACEHOLDERS (MUST EXIST AT RUNTIME) ---
# llm = ...
# original_text = ...
# bundle_json = ...

# --- ABDM CORRECTION PROMPT (hardcoded) ---
# correction_prompt = f"""
# SYSTEM: You are ABDM FHIR VALIDATION EXPERT. Return ONLY the corrected complete ABDM Bundle JSON.

# MANDATORY FIXES (in order):
# 1. Composition FIRST (entry[0], resourceType="Composition", correct NDHM profile)
# 2. DocumentReference LAST (all DocumentReference at end)  
# 3. ALL REFERENCES RESOLVE (urn:uuid: refs exist in bundle.entry)
# 4. TERMINOLOGY: LOINC(http://loinc.org), SNOMED(http://snomed.info/sct), UCUM(http://unitsofmeasure.org)
# 5. PROFILES: https://nrces.in/ndhm/fhir/r4/StructureDefinition/*
# 6. CONTENT: All clinical values from original_text present
# 7. MANDATORY: status, subject, issued, effectiveDateTime
# 8. NO DATA LOSS - only fix/add, never delete clinical content

# INPUT:
# ORIGINAL TEXT (SOURCE):
# {original_text}

# BUNDLE TO CORRECT:
# {json.dumps(bundle_json, indent=2)}

# RULES:
# - Return ONLY valid JSON Bundle (no markdown, no explanations)
# - Print "FIXED: description" for each change made
# - First entry MUST be Composition
# - Last entries MUST be DocumentReference  
# - All references must resolve within bundle
# - Preserve ALL clinical data from original_text

# OUTPUT ONLY THE CORRECTED JSON BUNDLE:
# """


correction_prompt = f"""
SYSTEM: You are ABDM FHIR VALIDATION EXPERT. Return ONLY the corrected complete ABDM Bundle JSON.


INPUT:
ORIGINAL TEXT (SOURCE):
{original_text}

BUNDLE TO CORRECT:
{json.dumps(bundle_json, indent=2)}


MANDATORY FIXES (in order):
1. Composition FIRST (entry[0], resourceType="Composition", correct NDHM profile)
2. DocumentReference LAST (all DocumentReference at end)  
3. ALL REFERENCES RESOLVE (urn:uuid: refs exist in bundle.entry)
4. TERMINOLOGY: LOINC(http://loinc.org), SNOMED(http://snomed.info/sct), UCUM(http://unitsofmeasure.org)
5. PROFILES: https://nrces.in/ndhm/fhir/r4/StructureDefinition/*
6. CONTENT: All clinical values from original_text present
7. MANDATORY: status, subject, issued, effectiveDateTime
8. NO DATA LOSS - only fix/add, never delete clinical content

STRICT DATA PERSISTENCE RULES:
- ZERO DELETION POLICY: You are strictly forbidden from removing any Observation, Condition, or Clinical resource present in the "BUNDLE TO CORRECT". 
- SURGICAL REPAIR ONLY: If a resource is invalid, fix its structure, add missing mandatory fields (like 'system' or 'code'), and align it with the profile. DO NOT discard the resource.
- AUGMENTATION: If the "ORIGINAL TEXT" contains clinical information (e.g., blood pressure, symptoms, dates) missing from the "BUNDLE TO CORRECT", you MUST add them as new Observations or fill in the empty fields.
- OBSERVATION INTEGRITY: Ensure every 'valueQuantity' or 'valueCodeableConcept' from the original input is preserved. If the validator says a field is "extra" or "not allowed", move that data into a 'note' or 'comment' field rather than deleting it.


RULES:
- Return ONLY valid JSON Bundle (no markdown, no explanations)
- First entry MUST be Composition
- Last entries MUST be DocumentReference  
- All references must resolve within bundle
- Preserve ALL clinical data from original_text and ALL existing resources from the input JSON.

OUTPUT ONLY THE CORRECTED JSON BUNDLE:
"""


# ✅ RUN CORRECTION
print("🔍 Starting ABDM Bundle Correction...")

result = llm.invoke(correction_prompt)

print("\n" + "=" * 80)
print("🎯 CORRECTION PROCESS COMPLETE")
print("=" * 80)


🔍 Starting ABDM Bundle Correction...

🎯 CORRECTION PROCESS COMPLETE
❌ Unexpected error: 'list' object has no attribute 'split'

Raw LLM Output:
```json
{
  "resourceType": "Bundle",
  "id": "9ecc9fdf-c81d-4f26-93f0-2ada2b4a4173",
  "meta": {
    "profile": [
      "https://nrces.in/ndhm/fhir/r4/StructureDefinition/DocumentBundle"
    ]
  },
  "type": "document",
  "identifier": {
    "system": "https://www.abdm.gov.in/bundle",
    "value": "59277708-fdae-46e3-94a3-083643163ee2"
  },
  "timestamp": "2026-02-26T10:50:43.899369+00:00",
  "entry": [
    {
      "fullUrl": "urn:uuid:d5b9c3e7-6f4a-4128-b0c5-8b9c7a3e9f4a",
      "resource": {
        "resourceType": "Composition",
        "id": "d5b9c3e7-6f4a-4128-b0c5-8b9c7a3e9f4a",
        "status": "final",
        "type": {
          "coding": [
            {
              "system": "http://loinc.org",
              "code": "34133-9",
              "display": "Discharge summary"
            }
          ],
          "text": "Discharge Summ

In [ ]:
def extract_json(text: str):
    if not text or not text.strip():
        return None
    
    # Remove markdown code blocks
    text = text.strip()
    if text.startswith("```json"):
        text = text[7:]
    if text.startswith("```"):
        text = text[3:]
    if text.endswith("```"):
        text = text[:-3]
    text = text.strip()
    
    decoder = json.JSONDecoder()
    idx = 0
    
    while idx < len(text):
        try:
            obj, end = decoder.raw_decode(text[idx:])
            if isinstance(obj, str):
                try:
                    obj = json.loads(obj)
                except:
                    pass
            return obj
        except json.JSONDecodeError:
            idx += 1
    return None


# --- EXTRACT AND SAVE FINAL BUNDLE ---
try:
    # Clean any markdown code blocks
    json_obj = extract_json(result.content)

    if json_obj is None:
        raise ValueError("No valid JSON found in model output")

    final_bundle = json_obj
    
    # Save validated bundle
    with open("validated_abdm_bundle_discharge_summary.json", "w") as f:
        json.dump(final_bundle, f, indent=2)
    
    print("Json is Extracted well....")
    entries = final_bundle.get('entry', [])
    comp_type = entries['resource']['resourceType'] if entries else "Empty"
    last_type = entries[-1]['resource']['resourceType'] if entries else "Empty"
    
    print(f"✅ SAVED: validated_abdm_bundle_discharge_summary.json")
    print(f"📊 Total Entries: {len(entries)}")
    print(f"🎯 First Resource: {comp_type} (Should be Composition)")
    print(f"📄 Last Resource: {last_type} (Should be DocumentReference)")
    
    print("\n📋 Bundle Structure OK ✅")
    
except json.JSONDecodeError as e:
    print("❌ JSON parsing failed")
    print(f"Error: {e}")
    print("\nRaw LLM Output:")
    print(result.content)
    
except Exception as e:
    print(f"❌ Unexpected error: {e}")
    print("\nRaw LLM Output:")
    print(result.content)


❌ Unexpected error: list indices must be integers or slices, not str

Raw LLM Output:
```json
{
  "resourceType": "Bundle",
  "id": "9ecc9fdf-c81d-4f26-93f0-2ada2b4a4173",
  "meta": {
    "profile": [
      "https://nrces.in/ndhm/fhir/r4/StructureDefinition/DocumentBundle"
    ]
  },
  "type": "document",
  "identifier": {
    "system": "https://www.abdm.gov.in/bundle",
    "value": "59277708-fdae-46e3-94a3-083643163ee2"
  },
  "timestamp": "2026-02-26T10:50:43.899369+00:00",
  "entry": [
    {
      "fullUrl": "urn:uuid:d5b9c3e7-6f4a-4128-b0c5-8b9c7a3e9f4a",
      "resource": {
        "resourceType": "Composition",
        "id": "d5b9c3e7-6f4a-4128-b0c5-8b9c7a3e9f4a",
        "status": "final",
        "type": {
          "coding": [
            {
              "system": "http://loinc.org",
              "code": "34133-9",
              "display": "Discharge summary"
            }
          ],
          "text": "Discharge Summary"
        },
        "subject": {
          "reference"

In [3]:
import subprocess

cmd = [
    "java",
    "-jar",
    "validator_cli.jar",
    "dynamic_abdm_bundle_test_1_diagnosticReport.json",
    "-version",
    "4.0.1"
]

try:
    # Adding check=True will raise an exception if the validator fails
    result = subprocess.run(cmd, capture_output=True, text=True, check=False)

    print("--- VALIDATION OUTPUT ---")
    print(result.stdout)

    if result.stderr:
        print("--- SYSTEM/JAVA ERRORS ---")
        print(result.stderr)

except FileNotFoundError:
    print("Error: 'java' is not installed or 'validator_cli.jar' is missing from this directory.")

--- VALIDATION OUTPUT ---
FHIR Validation tool Version 6.8.1 (Git# 9ca7d6530e65). Built 2026-02-19T17:11:00.975Z (6 days old)
  Java:   17.0.18 from /usr/lib/jvm/java-17-openjdk-amd64 on amd64 (64bit). 30688MB available
  Paths:  Current = /media/bharath/DATA_8TB1/Bharath/Problem_Statement_2_630a8c8cb6, Package Cache = /home/bharath/.fhir/packages
  Params: dynamic_abdm_bundle_test_1_diagnosticReport.json -version 4.0.1
  Locale: India/IN
  Jurisdiction: India
Loading
  Loading FHIR v4.0.1 from hl7.fhir.r4.core#4.0.1
  Load hl7.terminology.r4#6.2.0 - 4288 resources (00:05.120)
  Load hl7.fhir.uv.extensions.r4#5.2.0 - 759 resources (00:01.217)
  Loaded FHIR - 8265 resources (00:00.000)
  Terminology server http://tx.fhir.org - Version Connected to Terminology Server at http://tx.fhir.org (00:05.994)
  Load hl7.fhir.uv.extensions.r5#5.2.0 - 759 resources (00:02.587)
  Load hl7.terminology#7.0.1 - 4066 resources (00:00.579)
  Load hl7.terminology.r5#6.5.0 - 4357 resources (00:00.716)
  Lo

In [20]:
import subprocess
import json
import os

# --- 1. RUN THE HL7 VALIDATOR ---
print("🔬 Step 1: Running HL7 FHIR Validator...")

# We use the IG for ABDM to ensure the validator knows the Indian profiles
# Replace 'nrces.in.ndhm#6.0.0' with your specific version if different
to_validate_path = "validated_abdm_bundle_discharge_summary.json"
healed_path = "healed_abdm_bundle_discharge_summary.json"

cmd = [
    "java", "-Xmx2G", "-jar", "validator_cli.jar",
    to_validate_path, 
    "-version", "4.0.1",
    "-ig", "nrces.in.ndhm#6.0.0" 
]

validator_result = subprocess.run(cmd, capture_output=True, text=True)
validation_report = validator_result.stdout

print("--- VALIDATOR FEEDBACK CAPTURED ---")

with open(to_validate_path, 'r') as f:
    bundle_json = json.load(f)

# --- 2. PREPARE THE POWERFUL REPAIR PROMPT ---
# We combine the Original Text, the Flawed JSON, and the Error Report
repair_prompt = f"""
### ROLE
You are an ABDM FHIR Surgical Repair Agent. Your goal is to fix validation errors WITHOUT deleting any clinical data.

### INPUT DATA
1. SOURCE CLINICAL TEXT: 
{original_text}

2. FAILED BUNDLE (JSON):
{json.dumps(bundle_json, indent=2)}

3. VALIDATOR ERROR LOG:
{validation_report}

### MANDATORY REPAIR RULES
- **STRICT NON-DELETION**: Do NOT remove any Observation components, clinical values, or extensions. If a field has an error, FIX the format or the profile reference; NEVER delete the field.
- **ABDM COMPLIANCE**: Ensure the first entry is a Composition. Fix missing 'status', 'subject', or 'system' attributes.
- **REFERENCE INTEGRITY**: Ensure all 'urn:uuid' references are internally consistent.
- **TERMINOLOGY**: If the validator flags a code as invalid, find the correct mapping from the Source Text or use the validator's suggestion, but keep the 'display' text intact.

### OUTPUT INSTRUCTIONS
- Return ONLY the raw JSON Bundle. 
- No preamble, no markdown code blocks, no "Here is your fixed JSON".
- If a field is unknown to you but exists in the input JSON, LEAVE IT UNTOUCHED.

### OUTPUT CORRECTED JSON:
"""

# --- 3. INVOKE LLM FOR CORRECTION ---
print("🤖 Step 2: LLM is repairing the Bundle based on Validator Errors...")
correction_response = llm.invoke(repair_prompt)

# --- 4. PARSE AND SAVE THE HEALED BUNDLE ---
try:
    # Extract JSON from potential Markdown formatting
    raw_content = correction_response.content.strip()
    if "```json" in raw_content:
        clean_json = raw_content.split("```json")[1].split("```")[0].strip()
    elif "```" in raw_content:
        clean_json = raw_content.split("```")[1].split("```")[0].strip()
    else:
        clean_json = raw_content

    healed_bundle = json.loads(clean_json)
    
    with open(healed_path, "w") as f:
        json.dump(healed_bundle, f, indent=2)
        
    print(f"\n✅ SUCCESS: Healed bundle saved to {healed_path}")
    
    # Quick Structure Check
    first_res = healed_bundle['entry'][0]['resource']['resourceType']
    print(f"📊 New Bundle Entry Count: {len(healed_bundle.get('entry', []))}")
    print(f"🎯 First Resource check: {first_res}")

except Exception as e:
    print(f"❌ Failed to parse healed JSON: {e}")
    print("Full LLM Output for debugging:")
    print(correction_response.content)

🔬 Step 1: Running HL7 FHIR Validator...
--- VALIDATOR FEEDBACK CAPTURED ---
🤖 Step 2: LLM is repairing the Bundle based on Validator Errors...

✅ SUCCESS: Healed bundle saved to healed_abdm_bundle_discharge_summary.json
📊 New Bundle Entry Count: 17
🎯 First Resource check: Composition


In [22]:
import subprocess
import json
import os
from typing import TypedDict, Annotated, Sequence
from langgraph.graph import StateGraph, END
from langgraph.prebuilt import ToolNode
from langchain_core.messages import HumanMessage, AIMessage
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import JsonOutputParser
import operator

# State definition for the workflow
class AgentState(TypedDict):
    bundle_json: dict
    validation_report: str
    original_text: str
    iteration: int
    max_iterations: int
    healed_path: str
    success: bool

# Initialize state
def init_state():
    to_validate_path = "validated_abdm_bundle_discharge_summary.json"
    healed_path = "healed_abdm_bundle_discharge_summary_3agents.json"
    
    with open(to_validate_path, 'r') as f:
        initial_bundle = json.load(f)
    
    # with open("original_clinical_text.txt", 'r') as f:  # Assume you have this file
    #     original_text = f.read()
    
    return {
        "bundle_json": initial_bundle,
        "validation_report": "",
        "original_text": original_text,
        "iteration": 0,
        "max_iterations": 3,
        "healed_path": healed_path,
        "success": False
    }

# Step 1: HL7 FHIR Validator Agent
def validator_agent(state: AgentState) -> AgentState:
    print(f"🔬 Iteration {state['iteration'] + 1}: Running HL7 FHIR Validator...")
    
    # Save current bundle for validation
    temp_validate_path = f"temp_validate_iter_{state['iteration']}.json"
    with open(temp_validate_path, "w") as f:
        json.dump(state['bundle_json'], f, indent=2)
    
    # Run validator
    cmd = [
        "java", "-Xmx2G", "-jar", "validator_cli.jar",
        temp_validate_path,
        "-version", "4.0.1",
        "-ig", "nrces.in.ndhm#6.0.0"
    ]
    
    validator_result = subprocess.run(cmd, capture_output=True, text=True)
    validation_report = validator_result.stdout
    
    print("✅ Validator feedback captured")
    
    # Update state
    state['validation_report'] = validation_report
    state['iteration'] += 1
    
    return state

# Step 2: LLM Surgical Repair Agent (same prompt for all iterations)
repair_prompt_template = """
### ROLE
You are an ABDM FHIR Surgical Repair Agent. Your goal is to fix validation errors WITHOUT deleting any clinical data.

### INPUT DATA
1. SOURCE CLINICAL TEXT: 
{original_text}

2. FAILED BUNDLE (JSON):
{failed_bundle}

3. VALIDATOR ERROR LOG:
{validation_report}

### MANDATORY REPAIR RULES
- **STRICT NON-DELETION**: Do NOT remove any Observation components, clinical values, or extensions. If a field has an error, FIX the format or the profile reference; NEVER delete the field.
- **ABDM COMPLIANCE**: Ensure the first entry is a Composition. Fix missing 'status', 'subject', or 'system' attributes.
- **REFERENCE INTEGRITY**: Ensure all 'urn:uuid' references are internally consistent.
- **TERMINOLOGY**: If the validator flags a code as invalid, find the correct mapping from the Source Text or use the validator's suggestion, but keep the 'display' text intact.

### OUTPUT INSTRUCTIONS
- Return ONLY the raw JSON Bundle.
- No preamble, no markdown code blocks, no "Here is your fixed JSON".
- If a field is unknown to you but exists in the input JSON, LEAVE IT UNTOUCHED.

### OUTPUT CORRECTED JSON:
"""

def repair_agent(state: AgentState) -> AgentState:
    print(f"🤖 Iteration {state['iteration']}: LLM repairing Bundle...")
    
    # Format prompt
    prompt = repair_prompt_template.format(
        original_text=state['original_text'],
        failed_bundle=json.dumps(state['bundle_json'], indent=2),
        validation_report=state['validation_report']
    )
    
    # Invoke LLM
    messages = [HumanMessage(content=prompt)]
    correction_response = llm.invoke(messages)
    
    # Parse JSON response
    try:
        raw_content = correction_response.content.strip()
        if "```json" in raw_content:
            clean_json = raw_content.split("```json").split("```").strip()[1]
        elif "```" in raw_content:
            clean_json = raw_content.split("```").split("```")[0].strip()
        else:
            clean_json = raw_content
        
        healed_bundle = json.loads(clean_json)
        state['bundle_json'] = healed_bundle
        
        print(f"✅ Iteration {state['iteration']}: Bundle repaired successfully")
        
    except Exception as e:
        print(f"❌ Iteration {state['iteration']}: Failed to parse JSON: {e}")
        print("LLM Output:", correction_response.content[:500])
    
    return state

# Conditional edge: Check if we should continue
def should_continue(state: AgentState) -> str:
    if state['iteration'] >= state['max_iterations']:
        return "final_save"
    
    # Quick check: If first resource is Composition and no critical errors, we can stop early
    try:
        first_entry = state['bundle_json']['entry'][0]['resource']
        if first_entry.get('resourceType') == 'Composition':
            print(f"🎯 Early stop: First resource is Composition at iteration {state['iteration']}")
            return "final_save"
    except:
        pass
    
    return "repair"

# Final save and validation
def final_save_agent(state: AgentState) -> AgentState:
    print(f"🏁 Final Save: Iteration {state['iteration']}")
    
    # Save final healed bundle
    with open(state['healed_path'], "w") as f:
        json.dump(state['bundle_json'], f, indent=2)
    
    # Final structure check
    try:
        entry_count = len(state['bundle_json'].get('entry', []))
        first_res = state['bundle_json']['entry'][0]['resource']['resourceType']
        print(f"📊 Final Bundle Entry Count: {entry_count}")
        print(f"🎯 Final First Resource: {first_res}")
        state['success'] = True
    except Exception as e:
        print(f"⚠️ Final structure check failed: {e}")
        state['success'] = False
    
    return state

# Build the LangGraph workflow
def create_workflow():
    workflow = StateGraph(AgentState)
    
    # Add nodes
    workflow.add_node("validate", validator_agent)
    workflow.add_node("repair", repair_agent)
    workflow.add_node("save", final_save_agent)
    
    # Set entry point
    workflow.set_entry_point("validate")
    
    # Add edges
    workflow.add_edge("validate", "repair")
    workflow.add_conditional_edges(
        "repair",
        should_continue,
        {
            "repair": "validate",
            "final_save": "save"
        }
    )
    workflow.add_edge("save", END)
    
    return workflow.compile()

# 🚀 RUN THE 3-AGENT WORKFLOW
print("🔬🔬🔬 ABDM FHIR 3-Agent Surgical Repair Pipeline Starting...")
print("=" * 60)

# Initialize and run
initial_state = init_state()
app = create_workflow()

final_state = app.invoke(initial_state)

print("\n" + "=" * 60)
print("🎉 PIPELINE COMPLETE!")
print(f"✅ Success: {final_state['success']}")
print(f"📁 Final file: {final_state['healed_path']}")
print(f"🔄 Total iterations: {final_state['iteration']}")


🔬🔬🔬 ABDM FHIR 3-Agent Surgical Repair Pipeline Starting...
🔬 Iteration 1: Running HL7 FHIR Validator...
✅ Validator feedback captured
🤖 Iteration 1: LLM repairing Bundle...
✅ Iteration 1: Bundle repaired successfully
🎯 Early stop: First resource is Composition at iteration 1
🏁 Final Save: Iteration 1
📊 Final Bundle Entry Count: 17
🎯 Final First Resource: Composition

🎉 PIPELINE COMPLETE!
✅ Success: True
📁 Final file: healed_abdm_bundle_discharge_summary_3agents.json
🔄 Total iterations: 1
